# Train Qwen3-4B ZAIC 2026 — Data `data_v7_final` & L4 Max Safe Config

Notebook huấn luyện **V7 Final** tối ưu an toàn cho **NVIDIA L4 (24GB VRAM)**:
1. **Dataset `data_v7_final` (`2,046` train / `201` val)**: Giữ 100% nền sạch `data_v6_adapted` (đã đạt `49.30000`) + **140 mẫu spoken/ASR** bổ sung đúng 14 app yếu nhất theo tỷ lệ khóa (`111 DOABLE / 20 ASK_INFO / 9 REJECT`). Tự động tải thẳng từ GitHub `sun0810pi/zalo-challenge` về Google Drive (không cần upload thủ công).
2. **Cô lập hoàn toàn khỏi V6**: Lưu riêng tại `/content/drive/MyDrive/zalo-challenge-v7` (tuyệt đối không ghi đè thư mục V6 cũ).
3. **Thông số Max an toàn cho L4 (Không cắt cụt & Không OOM)**:
   - **`MAX_SEQ_LEN = 9216`**: Tự đo phân bố token thật để đảm bảo **0% mẫu bị cắt mất `<tool_call>` cuối** (vượt qua giới hạn 8192 trong config mặc định).
   - **Native `bf16` + `tf32` + `sdpa` + `ChunkedLossTrainer` (block 256 tokens)** chống OOM cross-entropy.
   - **LoRA `r = 128, alpha = 256`**, `paged_adamw_8bit`, **`lr = 1.5e-4`**, `warmup_steps = 20`, `epochs = 3`.
4. **Auto Eval & Best Checkpoint (`eval_steps = 25`, `EarlyStopping(patience=4)`)**: Tránh overfit ở cuối epoch 3, tự nạp lại **best checkpoint** theo `eval_loss` thấp nhất và nén sẵn `.zip` vào Drive.

## Bước 1: Bật Allocator Chống Phân Mảnh CUDA, Kiểm tra GPU L4 & Gắn Google Drive

In [ ]:
import os
# Phải đặt TRƯỚC khi import torch để tránh giữ RAM phân mảnh (reserved unallocated)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
import torch

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/zalo-challenge-v7"
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

!nvidia-smi

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name} | VRAM: {vram_gb:.2f} GiB | Compute Capability: {props.major}.{props.minor} | bf16_supported={torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("Không tìm thấy GPU! Hãy vào Runtime -> Change runtime type -> Chọn NVIDIA L4 GPU.")

## Bước 2: Cài đặt Thư Viện Cần Thiết

In [ ]:
!pip install -q -U "transformers>=4.51.0" "datasets>=3.0.0" "peft>=0.14.0" "accelerate>=1.2.0" bitsandbytes
print("✅ Cài đặt thư viện hoàn tất!")

## Bước 3: Tự Động Nạp Dataset `data_v7_final` (Train `2,046` / Val `201`)
Ưu tiên dùng bản có sẵn trên Drive nếu đã tồn tại; nếu chưa có, tự tải trực tiếp từ GitHub `sun0810pi/zalo-challenge` (`main/data/sft/data_v7_final/`) và lưu cache tại `/content/drive/MyDrive/zalo-challenge-v7/data_v7_final/`.

In [ ]:
import os, json, zipfile, shutil, urllib.request
from pathlib import Path

CACHE_DATA_DIR = Path(PROJECT_DIR) / "data_v7_final"

CANDIDATE_DIRS = [
    CACHE_DATA_DIR,
    Path("/content/drive/MyDrive/data_v7_final"),
    Path("/content/drive/MyDrive/zalo-challenge/data/sft/data_v7_final"),
    Path("/content/data_v7_final"),
]
CANDIDATE_ZIPS = [
    Path(PROJECT_DIR) / "data_v7_final.zip",
    Path("/content/drive/MyDrive/data_v7_final.zip"),
    Path("/content/data_v7_final.zip"),
]

DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "sft_train.jsonl").exists() and (d / "sft_val.jsonl").exists()), None)

if DATA_DIR is None:
    zip_path = next((z for z in CANDIDATE_ZIPS if z.exists()), None)
    if zip_path is not None:
        CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Đang giải nén {zip_path} -> {CACHE_DATA_DIR.parent} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(CACHE_DATA_DIR.parent)
        if (CACHE_DATA_DIR / "sft_train.jsonl").exists() and (CACHE_DATA_DIR / "sft_val.jsonl").exists():
            DATA_DIR = CACHE_DATA_DIR
        else:
            for p in CACHE_DATA_DIR.parent.rglob("sft_train.jsonl"):
                if (p.parent / "sft_val.jsonl").exists():
                    DATA_DIR = p.parent
                    break

if DATA_DIR is None:
    GITHUB_RAW_BASE = "https://raw.githubusercontent.com/sun0810pi/zalo-challenge/main/data/sft/data_v7_final"
    CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for fname in ("sft_train.jsonl", "sft_val.jsonl", "manifest.json", "validation_report.json"):
        dest = CACHE_DATA_DIR / fname
        tmp = CACHE_DATA_DIR / f".tmp_{fname}"
        url = f"{GITHUB_RAW_BASE}/{fname}"
        print(f"Đang tải {url} -> {dest} ...")
        urllib.request.urlretrieve(url, tmp)
        shutil.move(str(tmp), str(dest))
    DATA_DIR = CACHE_DATA_DIR

TRAIN_FILE = str(DATA_DIR / "sft_train.jsonl")
VAL_FILE = str(DATA_DIR / "sft_val.jsonl")

n_train = sum(1 for l in open(TRAIN_FILE, encoding="utf-8") if l.strip())
n_val = sum(1 for l in open(VAL_FILE, encoding="utf-8") if l.strip())
print(f"✅ Dataset V7 verified at: {DATA_DIR}")
print(f"   - sft_train.jsonl: {n_train} dòng")
print(f"   - sft_val.jsonl  : {n_val} dòng")
assert n_train == 2046, f"Kỳ vọng 2046 mẫu train cho data_v7_final nhưng đọc được {n_train}!"
assert n_val == 201, f"Kỳ vọng 201 mẫu val cho data_v7_final nhưng đọc được {n_val}!"

if (DATA_DIR / "manifest.json").is_file():
    manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
    print("Manifest:", manifest.get("summary"))

## Bước 4: Cấu hình V7 L4 Max (`r=128`, `alpha=256`, `lr=1.5e-4`, `MAX_SEQ_LEN=9216`), Vá TorchAO & Collator

In [ ]:
# 1. Vá lỗi xung đột torchao trên Google Colab trước khi build LoRA
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

import gc
import math
import torch

# 2. Tự động nhận diện GPU & thiết lập thông số V7 Max an toàn
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
has_bf16 = torch.cuda.is_bf16_supported()

MODEL_ID = "Qwen/Qwen3-4B"
MAX_SEQ_LEN = 9216  # Đủ chứa trọn vẹn mẫu dài nhất (~28k ký tự) không cắt cụt tool_call

if vram_gb >= 35.0 and has_bf16:
    HW_PROFILE = "A100_MAX_BF16_V7"
    USE_4BIT = False
    LORA_R = 128
    LORA_ALPHA = 256
    PER_DEVICE_BATCH_SIZE = 2
    GRAD_ACCUM = 8
    OPTIM_NAME = "paged_adamw_8bit"
elif vram_gb >= 20.0 and has_bf16:
    HW_PROFILE = "L4_MAX_BF16_CHUNKED_CE_V7"
    USE_4BIT = False
    LORA_R = 128
    LORA_ALPHA = 256
    PER_DEVICE_BATCH_SIZE = 1
    GRAD_ACCUM = 16
    OPTIM_NAME = "paged_adamw_8bit"
else:
    HW_PROFILE = "T4_SAFE_QLORA_V7"
    USE_4BIT = True
    LORA_R = 64
    LORA_ALPHA = 128
    PER_DEVICE_BATCH_SIZE = 1
    GRAD_ACCUM = 16
    OPTIM_NAME = "paged_adamw_8bit"

SAFE_OVERRIDE_R = None
if SAFE_OVERRIDE_R is not None:
    LORA_R = int(SAFE_OVERRIDE_R)
    LORA_ALPHA = LORA_R * 2

NUM_EPOCHS = 3
LEARNING_RATE = 1.5e-4
WARMUP_STEPS = 20
EVAL_STEPS = 25
SAVE_STEPS = 25
EARLY_STOPPING_PATIENCE = 4

if has_bf16:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=== HARDWARE & V7 HYPERPARAMETER PROFILE ===")
print(f"Profile      : {HW_PROFILE} ({props.name}, {vram_gb:.2f} GiB)")
print(f"Precision    : {'Native BF16 + TF32' if not USE_4BIT else '4-bit NF4 QLoRA'} | Chunked CE (256 tok/chunk)")
print(f"LoRA Config  : r={LORA_R}, alpha={LORA_ALPHA}, dropout=0.05 (all 7 linear layers)")
print(f"Sequence Cap : MAX_SEQ_LEN={MAX_SEQ_LEN} (auto-verify 0 truncation)")
print(f"Batch / Opt  : bs={PER_DEVICE_BATCH_SIZE} x accum={GRAD_ACCUM} (eff=16) | {OPTIM_NAME} | lr={LEARNING_RATE} | warmup={WARMUP_STEPS}")
print(f"Schedule     : epochs={NUM_EPOCHS}, eval/save every {EVAL_STEPS} steps, early_stop={EARLY_STOPPING_PATIENCE}")

# 3. ChatML Formatter giữ nguyên thẻ <think>...</think> giống y hệt pipeline train/infer của repo
def render_chatml_preserving_think(tokenizer, messages, add_generation_prompt=False):
    _THINK_OPEN, _THINK_CLOSE = "__QWEN3_THINK_OPEN__", "__QWEN3_THINK_CLOSE__"
    masked_msgs = []
    for m in messages:
        content = m.get("content") or ""
        if m.get("role") == "assistant" and "</think>" in content:
            c = content.replace("<think>", _THINK_OPEN).replace("</think>", _THINK_CLOSE)
            masked_msgs.append({"role": "assistant", "content": c})
        else:
            masked_msgs.append(m)
    try:
        p = tokenizer.apply_chat_template(
            masked_msgs,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        p = tokenizer.apply_chat_template(
            masked_msgs,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    p = p.replace("<|im_start|>assistant\n<think>\n\n</think>\n\n", "<|im_start|>assistant\n")
    return p.replace(_THINK_OPEN, "<think>").replace(_THINK_CLOSE, "</think>")

# 4. Data Collator chỉ tính loss trên token của Assistant (từ sau <|im_start|>assistant\n đến hết <|im_end|>)
class AssistantOnlyDataCollator:
    def __init__(self, tokenizer, max_seq_len=9216):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.start_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
        self.end_ids = tokenizer.encode("<|im_end|>", add_special_tokens=False)

    def compute_labels(self, ids):
        labels = [-100] * len(ids)
        pos, n, m = 0, len(ids), len(self.start_ids)
        while pos < n:
            start_idx = -1
            for i in range(pos, n - m + 1):
                if ids[i:i + m] == self.start_ids:
                    start_idx = i
                    break
            if start_idx == -1:
                break
            content_start = start_idx + m
            end_idx = -1
            for i in range(content_start, n - len(self.end_ids) + 1):
                if ids[i:i + len(self.end_ids)] == self.end_ids:
                    end_idx = i
                    break
            end_token = n if end_idx == -1 else end_idx + len(self.end_ids)
            for j in range(content_start, end_token):
                labels[j] = ids[j]
            pos = end_token
        return labels

    def __call__(self, features):
        b_ids = [f["input_ids"][:self.max_seq_len] for f in features]
        b_labels = [f["labels"][:self.max_seq_len] for f in features]
        max_l = max(len(x) for x in b_ids)
        pad_to = ((max_l + 7) // 8) * 8
        pad_to = min(pad_to, self.max_seq_len)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else self.tokenizer.eos_token_id
        p_ids = [x + [pad_id] * (pad_to - len(x)) for x in b_ids]
        p_mask = [[1] * len(x) + [0] * (pad_to - len(x)) for x in b_ids]
        p_labels = [x + [-100] * (pad_to - len(x)) for x in b_labels]
        return {
            "input_ids": torch.tensor(p_ids, dtype=torch.long),
            "attention_mask": torch.tensor(p_mask, dtype=torch.long),
            "labels": torch.tensor(p_labels, dtype=torch.long),
        }

print("✅ Hoàn tất cấu hình V7 & Data Collator.")

## Bước 5: Tokenize, Kiểm Tra 0% Cắt Cụt (`MAX_SEQ_LEN`) & Nạp Mô Hình `Qwen/Qwen3-4B`

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

raw_dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})

def measure_lengths(batch):
    lens = []
    for msgs in batch["messages"]:
        txt = render_chatml_preserving_think(tokenizer, msgs, add_generation_prompt=False)
        lens.append(len(tokenizer.encode(txt, add_special_tokens=False)))
    return {"tok_len": lens}

len_ds = raw_dataset.map(measure_lengths, batched=True, batch_size=64)
all_lens = list(len_ds["train"]["tok_len"]) + list(len_ds["validation"]["tok_len"])
max_observed_len = int(max(all_lens))
p95_len = int(np.percentile(all_lens, 95))
p99_len = int(np.percentile(all_lens, 99))
over_8192 = sum(1 for x in all_lens if x > 8192)
print(f"Token length stats (train+val={len(all_lens)}): min={min(all_lens)}, p95={p95_len}, p99={p99_len}, max={max_observed_len}")
print(f"Số mẫu vượt 8192 tokens (sẽ bị cắt nếu dùng config mặc định): {over_8192}/{len(all_lens)}")

if max_observed_len > MAX_SEQ_LEN:
    old_cap = MAX_SEQ_LEN
    MAX_SEQ_LEN = ((max_observed_len + 127) // 64) * 64
    print(f"⚡ Tự nâng MAX_SEQ_LEN từ {old_cap} -> {MAX_SEQ_LEN} để 0 mẫu bị cắt cụt!")

collator = AssistantOnlyDataCollator(tokenizer, max_seq_len=MAX_SEQ_LEN)

def preprocess(sample):
    text = render_chatml_preserving_think(tokenizer, sample["messages"], add_generation_prompt=False)
    ids = tokenizer.encode(text, add_special_tokens=False)
    assert len(ids) <= MAX_SEQ_LEN, f"Overflow: {len(ids)} > {MAX_SEQ_LEN}"
    labels = collator.compute_labels(ids)
    n_sup = sum(1 for x in labels if x != -100)
    assert n_sup > 0, f"Sample {sample.get('id')} có 0 token assistant được train!"
    return {"input_ids": ids, "labels": labels, "length": len(ids), "supervised_tokens": n_sup}

tokenized_dataset = raw_dataset.map(
    preprocess,
    remove_columns=[c for c in raw_dataset["train"].column_names if c not in ("id",)],
)
print("✅ Tokenize hoàn tất 100%! Số mẫu train:", len(tokenized_dataset["train"]), "| Số mẫu val:", len(tokenized_dataset["validation"]))

s0 = tokenized_dataset["train"][0]
sup_text = tokenizer.decode([t for t, l in zip(s0["input_ids"], s0["labels"]) if l != -100], skip_special_tokens=False)
print("--- Kiểm tra Assistant-only labels sample[0] (220 ký tự đầu) ---")
print(sup_text[:220])
print("-----------------------------------------------------------------")

gc.collect()
torch.cuda.empty_cache()

load_kwargs = dict(
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True,
)
if USE_4BIT:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if has_bf16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    load_kwargs["torch_dtype"] = torch.bfloat16 if has_bf16 else torch.float16
else:
    load_kwargs["torch_dtype"] = torch.bfloat16 if has_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.config.use_cache = False

if USE_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM sau khi nạp model + LoRA: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

## Bước 6: Huấn Luyện V7 L4 Max (`ChunkedLossTrainer` Chống OOM) + Auto Eval & Best Checkpoint Selection
Lưu hoàn toàn vào `/content/drive/MyDrive/zalo-challenge-v7/final_model_v7_final_l4_max` và đóng gói `/content/drive/MyDrive/zalo-challenge-v7/qwen3_4b_zac_lora_v7_final_l4_max.zip`.

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
import inspect, os, json, shutil, torch, gc
import torch.nn.functional as F

gc.collect()
torch.cuda.empty_cache()

class ChunkedLossTrainer(Trainer):
    """Tránh materialize toàn bộ tensor logits [B, S, 151936] vào VRAM (nguyên nhân gây OOM 3.55 GiB)."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        causal_lm = model.get_base_model() if hasattr(model, "get_base_model") else model
        backbone = getattr(causal_lm, "model", None) or getattr(causal_lm, "transformer", None)
        lm_head = causal_lm.get_output_embeddings()

        outputs = backbone(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            use_cache=False,
            return_dict=True,
        )
        hidden_states = outputs.last_hidden_state

        shift_hidden = hidden_states[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        flat_h = shift_hidden.view(-1, shift_hidden.size(-1))
        flat_y = shift_labels.view(-1)

        valid_mask = flat_y != -100
        active_h = flat_h[valid_mask]
        active_y = flat_y[valid_mask]

        n_active = int(active_y.numel())
        if n_active == 0:
            loss = hidden_states.sum() * 0.0
            return (loss, outputs) if return_outputs else loss

        def _chunk_ce(h_chunk, y_chunk):
            if not torch.is_autocast_enabled() and h_chunk.dtype != lm_head.weight.dtype:
                h_chunk = h_chunk.to(lm_head.weight.dtype)
            logits_chunk = lm_head(h_chunk).float()
            return F.cross_entropy(logits_chunk, y_chunk, reduction="sum")

        chunk_size = 256
        total_loss = None
        for start in range(0, n_active, chunk_size):
            end = min(start + chunk_size, n_active)
            h_c = active_h[start:end]
            y_c = active_y[start:end]
            if model.training and h_c.requires_grad:
                c_loss = torch.utils.checkpoint.checkpoint(_chunk_ce, h_c, y_c, use_reentrant=False)
            else:
                c_loss = _chunk_ce(h_c, y_c)
            total_loss = c_loss if total_loss is None else (total_loss + c_loss)

        if num_items_in_batch is not None:
            denom = num_items_in_batch.to(total_loss.device) if torch.is_tensor(num_items_in_batch) else num_items_in_batch
            loss = total_loss / denom
        else:
            loss = total_loss / n_active

        return (loss, outputs) if return_outputs else loss

RUN_NAME = "v7_final_l4_max"
OUTPUT_DIR = f"{PROJECT_DIR}/checkpoints_{RUN_NAME}"
FINAL_MODEL_DIR = f"{PROJECT_DIR}/final_model_{RUN_NAME}"
ZIP_PATH = f"{PROJECT_DIR}/qwen3_4b_zac_lora_{RUN_NAME}.zip"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

valid_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
ta_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=5,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,
    group_by_length=True,
    length_column_name="length",
    bf16=(has_bf16),
    fp16=(not has_bf16),
    tf32=(has_bf16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=OPTIM_NAME,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    seed=42,
    report_to="none",
)

if "eval_strategy" in valid_ta_params:
    ta_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in valid_ta_params:
    ta_kwargs["evaluation_strategy"] = "steps"

ta_kwargs = {k: v for k, v in ta_kwargs.items() if k in valid_ta_params}
training_args = TrainingArguments(**ta_kwargs)

class VRAMLogCallback(TrainerCallback):
    def __init__(self, total_vram_gb):
        self.total_vram_gb = total_vram_gb
        self.logged = False

    def on_step_end(self, args, state, control, **kwargs):
        if not self.logged and state.global_step >= 1:
            self.logged = True
            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
            print(f"✅ Step 1 (batch dài nhất) an toàn! Peak VRAM: {peak_gb:.2f} / {self.total_vram_gb:.2f} GiB")

valid_trainer_params = set(inspect.signature(ChunkedLossTrainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        VRAMLogCallback(vram_gb),
    ],
)
if "processing_class" in valid_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in valid_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = ChunkedLossTrainer(**trainer_kwargs)

last_ckpt = get_last_checkpoint(OUTPUT_DIR)
if last_ckpt is not None:
    print(f"🔄 Phát hiện checkpoint V7 cũ, tiếp tục train từ: {last_ckpt}")
else:
    print("🔥 BẮT ĐẦU HUẤN LUYỆN V7 (L4 MAX + CHUNKED CE + AUTO BEST CHECKPOINT)...")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)
torch.cuda.empty_cache()
eval_metrics = trainer.evaluate()

print("\n=== TỔNG KẾT HUẤN LUYỆN V7 ===")
print("Best checkpoint :", trainer.state.best_model_checkpoint)
print("Best eval_loss  :", trainer.state.best_metric)
print("Final eval      :", eval_metrics)
print(f"Peak VRAM used  : {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")

history_path = os.path.join(FINAL_MODEL_DIR, "training_summary.json")
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
with open(history_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset": "data_v7_final",
        "n_train": n_train,
        "n_val": n_val,
        "hardware_profile": HW_PROFILE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "warmup_steps": WARMUP_STEPS,
        "max_seq_len": MAX_SEQ_LEN,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "final_eval": eval_metrics,
        "log_history": trainer.state.log_history,
    }, f, ensure_ascii=False, indent=2)

archive_base = ZIP_PATH[:-4] if ZIP_PATH.endswith(".zip") else ZIP_PATH
shutil.make_archive(archive_base, "zip", FINAL_MODEL_DIR)
print(f"✅ Đã lưu best V7 adapter tại: {FINAL_MODEL_DIR}")
print(f"✅ Đã nén sẵn file V7 zip tại  : {ZIP_PATH}")

## Bước 7: Sanity Check Inference Đúng Chuẩn Prompt Hệ Thống (`ALLOWED TOOLS`)

In [ ]:
import json, torch

model.eval()
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
model.config.use_cache = True

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = list(dict.fromkeys([t for t in [tokenizer.eos_token_id, im_end_id] if isinstance(t, int) and t >= 0]))

def predict_messages(title, msgs, expected=None):
    prompt = render_chatml_preserving_think(tokenizer, msgs, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=320,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_ids,
        )
    pred = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    print(f"\n=== {title} ===")
    if expected:
        print("Kỳ vọng  :", expected)
    print("Sinh ra  :", pred)

val_rows = [json.loads(l) for l in open(VAL_FILE, encoding="utf-8") if l.strip()]
val_by_id = {r["id"]: r for r in val_rows}

weak_val = val_by_id.get("v6_val_10_fuel_ron95_read") or next((r for r in val_rows if r["id"].startswith("v6_val_")), val_rows[0])
predict_messages(
    f"TEST 1: Held-out Val Weak Case ({weak_val['id']})",
    weak_val["messages"][:2],
    expected=weak_val["messages"][2]["content"].splitlines()[:5],
)

ask_val = next((r for r in val_rows if "sys.ask_missing_info" in r["messages"][2]["content"]), val_rows[0])
predict_messages(
    f"TEST 2: Held-out Val Missing Info ({ask_val['id']})",
    ask_val["messages"][:2],
    expected=ask_val["messages"][2]["content"].splitlines()[:5],
)

marker = "### DANH SÁCH ỨNG DỤNG ĐÃ CÀI ĐẶT & CÔNG CỤ ĐƯỢC PHÉP GỌI (ALLOWED TOOLS):"
sys_prefix = val_rows[0]["messages"][0]["content"].split(marker)[0] + marker + "\n\n"
sys_cancel = sys_prefix + "\n".join([
    "[Ứng dụng: Chứng khoán (stocks) | Nhóm: finance]",
    "Mô tả: Tra cứu giá, danh mục và đặt lệnh mua/bán cổ phiếu",
    json.dumps({"name": "stocks.place_order", "kind": "write", "desc": "Đặt lệnh mua/bán", "params": {"ticker": {"type": "string", "required": True}, "side": {"type": "string", "required": True}, "shares": {"type": "integer", "required": True}, "order_type": {"type": "string", "required": True}}}, ensure_ascii=False),
    "",
    "[Ứng dụng: Hệ thống (sys) | Nhóm: system]",
    "Mô tả: Hỏi lại người dùng khi thiếu thông tin",
    json.dumps({"name": "sys.ask_missing_info", "kind": "system", "desc": "Hỏi lại người dùng", "params": {"tool_name": {"type": "string", "required": True}, "params": {"type": "array", "required": True}}}, ensure_ascii=False),
])
user_cancel = "### LỊCH SỬ HỘI THOẠI TRƯỚC ĐÓ (ĐÃ THỰC HIỆN XONG TRONG STATE)\nKhông có lịch sử trước đó.\n\n### YÊU CẦU HIỆN TẠI CẦN XỬ LÝ\nNgười dùng: Mua giúp tôi 200 cổ phiếu FPT nha. À thôi để coi hết phiên nay đã, đừng đặt lệnh."
predict_messages(
    "TEST 3: Tự hủy giữa chừng (kỳ vọng KHÔNG gọi tool)",
    [{"role": "system", "content": sys_cancel}, {"role": "user", "content": user_cancel}],
)

## Bước 8 (Tùy Chọn): Merge LoRA V7 + Base Model Sang BF16 Để Nộp Docker
Sáng mai sau khi test infer V7 thấy điểm Public **vượt 49.3**, đặt `RUN_MERGE = True` để xuất sẵn full merged weights BF16 (~8 GB) vào Drive cho container Docker.

In [ ]:
RUN_MERGE = False

if RUN_MERGE:
    import gc, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    MERGED_DIR = f"{PROJECT_DIR}/merged_model_{RUN_NAME}"
    for obj_name in ("model", "trainer"):
        if obj_name in globals():
            del globals()[obj_name]
    gc.collect()
    torch.cuda.empty_cache()

    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True)
    merged = PeftModel.from_pretrained(base, FINAL_MODEL_DIR).merge_and_unload()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True).save_pretrained(MERGED_DIR)
    print(f"✅ Đã merge & lưu mô hình full BF16 tại: {MERGED_DIR}")
else:
    print("ℹ️ Bỏ qua bước merge (đặt RUN_MERGE = True khi điểm Public V7 vượt V6 để xuất model cho Docker).")